# Data Preprocessing of DEID_RX_ORDER.csv using SQL

In [1]:
"""
Created on Thursday Nov 22 2025

@author: Laura Rueda
"""

import pandas as pd
import os
import duckdb
import re
import gc

# 1. Cleaning and processing csv with duckdB

duckdb is a python library that doesn't need server installation and works inside the notebook but with the power of sql: "In-process OLAP database".


In [5]:
input_file_sql = 'data/deid_rx_order.csv'        # Input file path
output_file_sql = 'data/deid_rx_order_final_sql.csv'      # Final clean file

In [6]:
# --- ALTERNATIVE EFFICIENT METHOD USING DUCKDB ---
print("Using SQL (DuckDB) to clean, order and save the data...")

# Connect to the temporary database
con = duckdb.connect()
print("   DuckDB connected.")

# This query does ALL the work of Pandas but without using RAM:
# 1. Reads the temporary file
# 2. Fills nulls (COALESCE)
# 3. Orders by PATIENT_ID and Shifted_date
# 4. Saves the final result directly to disk
query = f"""
COPY (
    SELECT 
        AIM_GROUP, PATIENT_ID, ENCOUNTER_ID, RX_CODE, RX_NAME,
        COALESCE(ROUTE, 'Unknown') AS ROUTE,
        COALESCE(DOSE, 'Unknown') AS DOSE,
        COALESCE(FREQUENCY, 'Unknown') AS FREQUENCY,
        CAST(Shifted_date AS TIMESTAMP) AS Shifted_date
    FROM '{input_file_sql}'
    ORDER BY PATIENT_ID, Shifted_date
) TO '{output_file_sql}' (HEADER, DELIMITER ',')
"""

con.execute(query)
print(f"¡Ready! Clean file saved at: {output_file_sql}")

# --- IMPORTANT (Top 50) ---
# Since DuckDB saves directly to disk, the variable 'df' does not exist in memory.
# We load only the DOSE column so that your next Top 50 cell works:
print("Loading DOSE column for analysis...")
df = pd.read_csv(output_file_sql, usecols=['DOSE'])

Using SQL (DuckDB) to clean, order and save the data...
   DuckDB connected.
¡Ready! Clean file saved at: data/deid_rx_order_final_sql.csv
Loading DOSE column for analysis...


# 2. One-Hot-Encoding of DOSE column

#### 2.1 Normalization 

1st way: 79.71%

In [7]:
def normalize_dose_smart(value):
    if pd.isna(value) or value == 'Unknown':
        return 'Unknown'
    
    # Lowercase and strip spaces
    value = str(value).lower().strip()
    
    # 2. Remove thousand separators ("1,000 mg" -> "1000 mg")
    value = value.replace(',', '')
    
    # 3. REGULAR EXPRESSIONS: It looks for a number (integer or decimal) followed by anything
    match = re.match(r"(\d*\.?\d+)\s*([a-z]+.*)", value)
    
    if match:
        # match.group(1) is the number, match.group(2) is the unit
        number = match.group(1)
        unit = match.group(2).strip()
        
        # 4. UNIT STANDARDIZATION 
        unit_mapping = {
            'gm': 'g', 'gms': 'g', 'gram': 'g', 'grams': 'g',
            'mg': 'mg', 'mgs': 'mg', 'milligrams': 'mg',
            'mcg': 'mcg', 'micrograms': 'mcg',
            'ml': 'ml', 'mls': 'ml',
            'unit': 'units', 'u': 'units',
            'tablet': 'tab', 'tablets': 'tab', 'pill': 'tab'
        }
        
        clean_unit = unit_mapping.get(unit, unit)
        
        # 5. Rebuild clean string: "NUMBER + SPACE + UNIT"
        return f"{number} {clean_unit}"
        
    return value # Returns the original if it doesn't match the pattern (e.g., "one time")


print("Applying smart cleaning (Regex)...")
df['DOSE_CLEAN'] = df['DOSE'].astype(str).apply(normalize_dose_smart)

# Compare some examples before and after cleaning
print("\nExamples of improvements:")
print(df[['DOSE', 'DOSE_CLEAN']].sample(10))

Applying smart cleaning (Regex)...

Examples of improvements:
             DOSE DOSE_CLEAN
18052336      1 %        1 %
13653476  2 Units    2 units
14588122     2 mg       2 mg
15471330     3 MU       3 mu
7890748   100 mcg    100 mcg
7413636     81 mg      81 mg
6351483    100 mg     100 mg
5847593   Unknown    Unknown
17093877   100 mg     100 mg
4221579    50 mcg     50 mcg


2nd way: 83.00%

In [25]:
def normalize_dose_math(value):
    if pd.isna(value) or value == 'Unknown':
        return 'Unknown'
    
    # 1. Basic cleaning
    value = str(value).lower().strip()
    value = value.replace(',', '') # Remove thousand separators
    
    # 2. Regex to separate Number and Unit
    match = re.match(r"(\d*\.?\d+)\s*([a-z]+.*)", value)
    
    if match:
        try:
            number = float(match.group(1)) # Convert to real number for operations
            unit = match.group(2).strip()
            
            # 3. Standardize unit names (Singular/Plural/Synonyms)
            unit_mapping = {
                'gm': 'g', 'gms': 'g', 'gram': 'g', 'grams': 'g',
                'mg': 'mg', 'mgs': 'mg', 'milligrams': 'mg',
                'mcg': 'mcg', 'micrograms': 'mcg',
                'ml': 'ml', 'mls': 'ml',
                'l': 'l', 'liter': 'l', 'liters': 'l',
                'unit': 'units', 'u': 'units',
                'tablet': 'tab', 'tablets': 'tab', 'pill': 'tab', 'cap': 'cap', 'capsule': 'cap'
            }
            unit = unit_mapping.get(unit, unit)
            
            # 4. MATHEMATICAL CONVERSION (The trick to gain coverage)
            # Convert everything to the most common unit: mg and ml
            
            if unit == 'g':        # Grams to Milligrams
                number *= 1000
                unit = 'mg'
            elif unit == 'mcg':    # Micrograms to Milligrams
                number /= 1000
                unit = 'mg'
            elif unit == 'l':      # Liters to Milliliters
                number *= 1000
                unit = 'ml'
            elif unit == 'kg':     # Kilograms to Grams (rare in doses, but possible)
                number *= 1000
                unit = 'g'
                
            # 5. Formateo Inteligente (Quitar decimales innecesarios)
            # "1000.0 mg" se convierte en "1000 mg"
            if number.is_integer():
                clean_number = str(int(number))
            else:
                # Opcional: Redondear decimales locos (ej: 0.33333 -> 0.33)
                clean_number = str(round(number, 3))
                # Eliminar ceros finales '0.50' -> '0.5'
                if '.' in clean_number:
                     clean_number = clean_number.rstrip('0').rstrip('.')

            return f"{clean_number} {unit}"

        except ValueError:
            return value # Si falla la conversión matemática, devolvemos original

    return value


# --- PROBAR LA MEJORA ---
print("Aplicando normalización matemática avanzada...")
df['DOSE_CLEAN'] = df['DOSE'].astype(str).apply(normalize_dose_math)

# Recalcular el Top 50 con los nuevos datos
conteo = df['DOSE_CLEAN'].value_counts()
top_50_coverage = conteo.head(50).sum() / len(df) * 100

print(f"\nNueva Cobertura del TOP 50: {top_50_coverage:.2f}%")
print("\nEjemplos de fusiones conseguidas:")
# Ver casos donde cambió el original
cambios = df[df['DOSE'] != df['DOSE_CLEAN']][['DOSE', 'DOSE_CLEAN']].sample(10)
print(cambios)

Aplicando normalización matemática avanzada...

Nueva Cobertura del TOP 50: 83.96%

Ejemplos de fusiones conseguidas:
                              DOSE               DOSE_CLEAN
6482478                  1,000 mcg                     1 mg
8709931                5,000 units               5000 units
9877076                       2 gm                  2000 mg
14149319                  1,000 mg                  1000 mg
5531217                   1,000 mg                  1000 mg
12251017                  3.375 gm                  3375 mg
16852673                  1,750 mg                  1750 mg
16801194                     17 gm                 17000 mg
18790590  2,000 international unit  2000 international unit
4903046                    200 mcg                   0.2 mg


3rd way: 83.96%


In [8]:
def normalize_dose_ultimate(value):
    """
    Ultimate normalization: Handles mEq, sprays, puffs, and standard math.
    """
    if pd.isna(value) or value == 'Unknown':
        return 'Unknown'
    
    # 1. Basic Cleaning
    value = str(value).lower().strip()
    value = value.replace(',', '')
    
    # 2. Regex: Allow for number followed by potentially weird chars (like / or -)
    # This captures "number" + "space" + "unit"
    match = re.match(r"(\d*\.?\d+)\s*([a-z]+.*)", value)
    
    if match:
        try:
            number = float(match.group(1))
            unit = match.group(2).strip()
            
            # 3. MASTER MAPPING (The secret sauce)
            unit_mapping = {
                # Weight
                'gm': 'g', 'gms': 'g', 'gram': 'g', 'grams': 'g',
                'mg': 'mg', 'mgs': 'mg', 'milligrams': 'mg',
                'mcg': 'mcg', 'micrograms': 'mcg',
                'kg': 'kg', 'kilograms': 'kg',
                
                # Volume
                'ml': 'ml', 'mls': 'ml', 'milliliters': 'ml',
                'l': 'l', 'liter': 'l', 'liters': 'l',
                
                # Units / Potency
                'unit': 'units', 'u': 'units', 'iu': 'units', 'intl units': 'units',
                'meq': 'meq', 'mEq': 'meq', 'milliequivalents': 'meq', # KEEP THIS!
                
                # Forms
                'tablet': 'tab', 'tablets': 'tab', 'tab': 'tab', 'pill': 'tab', 'pills': 'tab',
                'cap': 'cap', 'capsule': 'cap', 'caps': 'cap',
                'patch': 'patch', 'patches': 'patch',
                'spray': 'spray', 'sprays': 'spray', 'puff': 'spray', 'puffs': 'spray',
                'app': 'app', 'appl': 'app', 'application': 'app',
                'vial': 'vial', 'amp': 'vial', 'ampul': 'vial'
            }
            
            # Normalize the unit name
            # We split by space to handle cases like "mg tablet" -> just "mg"
            first_word = unit.split()[0]
            unit = unit_mapping.get(first_word, first_word)
            
            # 4. MATH CONVERSIONS
            if unit == 'g': number *= 1000; unit = 'mg'
            elif unit == 'mcg': number /= 1000; unit = 'mg'
            elif unit == 'l': number *= 1000; unit = 'ml'
            elif unit == 'kg': number *= 1000; unit = 'g'
                
            # 5. Formatting
            if number.is_integer():
                clean_number = str(int(number))
            else:
                clean_number = str(round(number, 3))
                if '.' in clean_number:
                     clean_number = clean_number.rstrip('0').rstrip('.')

            return f"{clean_number} {unit}"

        except ValueError:
            return value

    return value

# --- EJECUTAR ---
print("Applying ULTIMATE normalization...")
df['DOSE_CLEAN'] = df['DOSE'].astype(str).apply(normalize_dose_ultimate)

# Check coverage
counts = df['DOSE_CLEAN'].value_counts()
top_50_coverage = counts.head(50).sum() / len(df) * 100
print(f"\nNew Coverage: {top_50_coverage:.2f}%")

Applying ULTIMATE normalization...

New Coverage: 84.20%


#### 2.2 Selecting only most frequent doses (top 50)

In [9]:
# 1. Calculate how many times each dosis appears
# value_counts() counts and orders from most to least frequent
conteo_dosis = df['DOSE_CLEAN'].value_counts()
# Save the complete list to a text file
conteo_dosis.to_string('complete_doses_list.txt')

print("Complete list of different doses saved in 'complete_doses_list.txt'.")

# 2. Show only the TOP 50
print("--- TOP 50 MOST FREQUENT DOSES ---")
display(conteo_dosis.head(50))

# 3. Calculate the coverage of these TOP 50 doses
total_rows = len(df)
rows_top_50 = conteo_dosis.head(50).sum()
print(f"\nThese 50 doses cover {rows_top_50} rows.")
print(f"They represent {(rows_top_50 / total_rows) * 100:.2f}% of all your data.")

Complete list of different doses saved in 'complete_doses_list.txt'.
--- TOP 50 MOST FREQUENT DOSES ---


DOSE_CLEAN
Unknown       4318435
100 mg        1048542
10 mg          817334
20 mg          671198
5000 units     631998
500 mg         618805
4 mg           499481
40 mg          458704
1 gtt          457050
5 mg           449280
50 mg          443673
1000 mg        410077
25 mg          407808
30 mg          406351
1 mg           344413
600 mg         323817
2 mg           293678
300 mg         282068
1 app          280456
325 mg         261670
2000 mg        257697
200 mg         227461
150 mg         181723
2.5 mg         178799
400 mg         178696
15 mg          159141
80 mg          158569
650 mg         147377
2 spray        144323
0.2 mg         144236
800 mg         143887
60 mg          142543
0.025 mg       133476
81 mg          130496
1 patch        123922
250 mg         114280
3375 mg        101377
0.1 mg         100248
0.5 mg          98155
0.05 mg         97476
0.4 mg          95907
17000 mg        95019
3 ml            94436
12.5 mg         91251
8 mg            82774


These 50 doses cover 17208284 rows.
They represent 84.20% of all your data.


#### 2.3 ONE-HOT-ENCODING

In [21]:
# --- STEP 1: DEFINE THE TOP 50 LIST ---
# We extract the names of the 50 most frequent doses from your cleaned column
top_50_doses = df['DOSE_CLEAN'].value_counts().nlargest(50).index

# --- STEP 2: REDUCE CARDINALITY (The "Other" Bucket) ---
# Logic: If the dose is in the Top 50, keep it. If not, replace it with 'Other'.
# We use numpy/pandas vectorization (.where) which is much faster than a loop.
df['DOSE_REDUCED'] = df['DOSE_CLEAN'].where(df['DOSE_CLEAN'].isin(top_50_doses), 'Other')

# Check that it worked (you should only see 51 unique values now)
print(f"Unique categories after reduction: {df['DOSE_REDUCED'].nunique()} (Target: 51)")

# --- STEP 3: GENERATE ONE-HOT ENCODING ---
# This creates the binary columns (0s and 1s).
# CRITICAL: dtype='int8' forces Python to use 1 byte per number instead of 8. 
# This reduces memory usage by 87% (Vital for 20 million rows).
dose_dummies = pd.get_dummies(df['DOSE_REDUCED'], prefix='DOSE', dtype='int8')

# --- STEP 4: PREVIEW ---
print("\nOne-Hot Matrix Shape:", dose_dummies.shape)
print(dose_dummies.head())

# NOTE: The 'dose_dummies' variable now holds your features for the Neural Network.
# Since you have 20M rows, avoid doing "pd.concat([df, dose_dummies])" unless you have >32GB RAM.
# It's safer to save 'dose_dummies' or use it directly in batches.

Unique categories after reduction: 51 (Target: 51)

One-Hot Matrix Shape: (20436701, 51)
   DOSE_0.025 mg  DOSE_0.05 mg  DOSE_0.1 mg  DOSE_0.2 mg  DOSE_0.4 mg  \
0              0             0            0            0            0   
1              0             0            0            0            0   
2              0             0            0            0            0   
3              0             0            0            0            0   
4              0             1            0            0            0   

   DOSE_0.5 mg  DOSE_1 app  DOSE_1 gtt  DOSE_1 mg  DOSE_1 patch  ...  \
0            0           0           0          0             0  ...   
1            0           0           0          0             0  ...   
2            0           0           0          0             0  ...   
3            0           0           0          1             0  ...   
4            0           0           0          0             0  ...   

   DOSE_600 mg  DOSE_650 mg  DOSE_75 mg

Dependencies needed for using Parquet:
- pip install pyarrow
- pip install fastparquet

In [22]:
# --- STEP 5: SAVE THE ONE-HOT ENCODING OF DOSES  ---
print("Saving dose matrix to disk (Parquet format)...")
#dose_dummies.to_pickle('data/dose_one_hot.pkl')
dose_dummies.to_parquet('data/dose_one_hot.parquet', engine='pyarrow')

print("Saved!")

# Free memory now that it's saved
del dose_dummies
gc.collect()
print("Memory freed.")

Saving dose matrix to disk (Parquet format)...
Saved!
Memory freed.
